<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/09-serving-inference/01-serving-frameworks.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 服务化框架（概念篇）

**目标：** 理解 AI 工程师实际要在其中做取舍的推理服务化技术栈（vLLM、TGI、Triton、TensorRT-LLM）：各自优化什么，它们如何映射到你一直在调用的那套原始 API，以及什么场景该用哪个。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 为什么先讲概念（以及什么能跑起来）

到目前为止的每个 notebook 都是通过 API 调用模型：Groq 托管模型，你发送 JSON。这一节讲的是**那套 API 另一侧的东西**——把权重加载到 GPU 上、把你的请求变成 token 的那台服务器。这一层，正是做 *AI 分布式系统* 的 AI 工程师所负责的。

Groq 教不了你这一层，因为它*本身*就是一个托管服务端点，没有东西可以让你搭起来。自托管 vLLM/Triton/TensorRT-LLM 又需要 GPU。所以本篇是**概念优先**：正文就是心智模型和决策依据（不需要 GPU），并在末尾附上一个**可选、且有明确围栏的附录**，为想看实物的人在免费的 Colab T4 上搭起一个真正的 vLLM 服务器。与 [第 06 节的 LoRA 附录](../06-adaptation/01-fine-tune-vs-rag-vs-prompt.zh.ipynb) 是同一套模式。

> **⭐ 关键结论 ——** 你一直以来都是推理服务器的*客户端*。服务化框架就是那台*服务器*。了解它们各自优化什么，你才能推理吞吐、延迟和成本——这正是 AI 系统面试真正会问的问题。

## 同一套 API，背后是两种截然不同的服务器

无论应答的是 Groq、OpenAI 还是你自己的机器，你发出的请求都一模一样：一次 OpenAI 兼容的 `chat.completions` 调用。不同的是请求**落地之后**发生的事：

```
你的请求 ─▶ [ 服务化框架 ] ─▶ GPU：加载权重、执行前向计算 ─▶ token ─▶ 你
                      ▲
            这就是第 09 节要讲的东西
```

服务化框架的全部职责，就是在*昂贵的*硬件上**高效**应答*大量*这样的请求。正是这一个约束（GPU 按秒计费、花的是真金白银，而单个请求几乎用不满它）驱动了下一篇里的每一项优化（batch、KV cache、量化）。这里我们先画出全貌。

## 你必须能说出口的四个名字

这些名字会出现在 AI 系统面试和招聘启事里。你不需要四个都跑过；你需要知道*每个是干什么用的*，以及*它们之间是什么关系*。

| 框架 | 谁做的 | 优化目标 | 一句话说明 |
|---|---|---|---|
| **vLLM** | UC Berkeley → 社区 | 通用 GPU 上的**吞吐** | 开源界的默认选择。PagedAttention 像操作系统虚拟内存那样管理 KV cache，因此单张 GPU 能服务多得多的并发请求。开箱即用提供 OpenAI 兼容服务器。 |
| **TGI**（Text Generation Inference） | Hugging Face | 简单、生产可用的服务化 | 开箱即全：metrics、safetensors、与 HF 生态贴合紧密。「把我的 HF 模型好好服务起来」的选项。 |
| **TensorRT-LLM** | NVIDIA | NVIDIA GPU 上的**最低延迟** | 把模型编译成硬件专属的 engine：单 token 最快，但构建步骤很重，且仅限 NVIDIA。 |
| **Triton Inference Server** | NVIDIA | 把*任何东西*服务化到规模 | 不专为 LLM；一个通用模型服务器（LLM、视觉、经典 ML），带 TensorRT-LLM backend。它是编排层，不是引擎。 |

> **💡 为什么这么多 ——** 它们处在不同层次，不是一个线性排名。Triton 是一个*服务器*，可以把 TensorRT-LLM *引擎*当作它的一个 backend 来跑；vLLM 和 TGI 则是自带引擎、自成一体的服务器。只有先说出指标，「最好」才有意义：吞吐、延迟，还是运维简单。

## 它如何映射到你已经搭好的循环

这里没有任何东西要取代你已知的部分；它是垫在*下面*的一层。回想 agent 循环（第 05 节）：你发送 `messages`，拿回响应，再发下一个请求。每一次调用都落在一个服务化框架上。把 Groq 换成自托管的 vLLM 服务器，你的代码里只有一处会变：

```python
# 第 01–05 节：Groq 的托管服务器
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=...)

# 同样的代码，换自托管 vLLM（启动方式：python -m vllm.entrypoints.openai.api_server --model ...）
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# ……下游的一切 —— tool calling、agent 循环、结构化输出 —— 逐字节完全相同。
```

这就是本仓库「OpenAI 兼容主干」的回报：**换供应商只需改 base URL，别的一概不动**。vLLM、TGI 和 Triton 都暴露同一套 OpenAI 兼容端点，所以你已经搭好的那些「不绑定框架」的模式，可以原样迁移到自托管部署，不用重写。

## AI 工程师什么时候用哪个

这个决策，在实际团队里的做法是这样的：

- **做原型 /「就想把这个 HF 模型服务起来」** → **TGI** 或 **vLLM**。设置最少，一条命令起一个 OpenAI 兼容服务器。
- **让每美元 GPU 承载的请求数最大化（大多数自托管 LLM 服务化场景）** → **vLLM**。它的连续批处理（continuous batching）+ PagedAttention 正是它成为社区默认的原因；这也是最常见的答案。
- **要榨干最后一个毫秒的延迟，硬件是 NVIDIA，愿意承担构建复杂度** → **TensorRT-LLM**（通常跑在 Triton *里面*）。
- **为多种模型类型（LLM + 视觉 + 经典 ML）提供统一服务化平台，多模型、带版本管理、面向整个组织** → **Triton**。
- **你只想要一个 API，完全不想碰 GPU** → 托管端点（Groq、OpenAI、Bedrock、Together、Fireworks）。大多数产品从这里起步，只有当成本或控制权提出要求时才自托管。

> **⚠️ 生产现实 ——** 自托管是成本/控制权的决策，不是默认选项。你换来规模下的更低单 token 成本、数据驻留或自定义模型的代价，是接手 GPU、自动扩缩容和 on-call。低于某个量级时，托管端点既更便宜*又*更省事。「我们一直用托管 API，直到量级足以支撑在 vLLM 上自托管」比「我们第一天就跑自己的 GPU」是更有力的面试回答。

关于系统层面的视角（这些服务器如何躲在负载均衡器后面、如何自动扩缩容、如何定容量），见下一节 [ML 系统设计](../10-ml-system-design/01-designing-an-inference-service.zh.ipynb)。

## 最佳实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 保留一个 OpenAI 兼容的客户端缝隙，切换服务层只是改配置 | 把某一家厂商的 SDK 写死，切换服务层就得重写 |
| 先说出指标（吞吐 / 延迟 / 运维简单），再做选择 | 心里没有指标就问「哪个框架最好？」 |
| 默认用托管端点；当成本/控制权/数据要求足够高时再自托管 | 因为「看起来更硬核」而第一天就自己上 GPU |
| 追求每美元吞吐就用 vLLM；追求最低延迟就用 TensorRT-LLM | 以为 TensorRT-LLM 只是「更快」，无视它的构建成本和 NVIDIA 锁定 |
| 明白 Triton 是能承载 TensorRT-LLM *引擎*的*服务器*（不同层次） | 把 Triton 和 TensorRT-LLM 当成两个相互竞争的引擎来比较 |

## 可选附录 —— 搭起一个真正的 vLLM 服务器（需要 GPU）

> **⚠️ 这个附录需要 GPU runtime，可以跳过。** 在 Colab 里：**Runtime → Change runtime type → T4 GPU**。它**不**用 Groq；它会下载一个小模型并在本地服务化。如果你用的是免费的 CPU runtime，读一读往下走即可；上面的概念才是要考察的内容。

这些单元格会安装 vLLM、在一个小模型上启动它的 OpenAI 兼容服务器，再用第 01 节*一模一样*的客户端代码去打它，以此证明：把客户端指向一个新的 base URL 就够了。**自托管是一次真实的依赖与资源演练，不是一键安装。** 这正是第 09 节的教训，所以下面这些单元格是故意*暴露*问题，而不是掩盖问题。你可能会撞上两个坑，这里都已处理：

- **import 时的 CUDA 版本冲突。** 安装 vLLM 会升级 PyTorch，这可能让 Colab 预装的 `torchaudio` 仍是按另一个 CUDA 版本编译的。而 `transformers` 会无条件 import `torchaudio`，于是这个不匹配会在*import 阶段*就让服务器崩溃。修法：卸掉 `torchaudio`（文本推理根本用不到它）。
- **「Free memory … is less than desired GPU memory utilization」。** 上一次尝试留下的 vLLM 进程还占着 GPU。同一时刻只能有一个 vLLM 拥有这张卡。修法：杀掉残留进程（或者 **Runtime → Restart session**），并调低 `--gpu-memory-utilization`。

In [ ]:
# 附录第 1 格 —— GPU 检查。如果这里打印出 "no GPU"，就到此为止（别往下跑）。
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "no GPU — switch runtime to T4, or skip this appendix (concepts above are what's graded)")

In [ ]:
# 附录第 2 格 —— 安装 vLLM（首次运行要几分钟；仅限 GPU），然后修掉 torchaudio 的
# CUDA 版本冲突。vLLM 会升级 PyTorch；Colab 里那份过期的 torchaudio 随后会 import 失败，进而
# 让服务器崩溃。文本推理用不到 torchaudio，所以我们把它卸掉。本来就不存在的话也无害。
%pip install -q vllm
!pip uninstall -y torchaudio 2>/dev/null || true
# 验证服务器需要的那些 import 确实能加载（这正是之前静默失败的地方）：
!python -c "import transformers, vllm; print('imports OK — vllm', vllm.__version__)"

# 在全新的 runtime 上，这个 import 检查会直接通过，不需要重启。少数情况下，如果 pip 升级了某个
# 已经加载进当前 kernel 的包、导致上面那行报错，就执行 Runtime -> Restart session，
# 然后从 GPU 检查那一格重新往下跑，跳过这个 %pip 行（包其实已经装好了）。

In [ ]:
# 附录第 3 格 —— 在后台启动这个 OpenAI 兼容服务器，并让日志可见。
# 几个关键选择，每一个都是踩坑踩出来的：
#   --enforce-eager             : 跳过 CUDA graph 编译（启动更快，还能避开一类 T4 崩溃）
#   --gpu-memory-utilization .85: 不要占 92%；留出余量，免得卡比较忙时启动失败
#   日志 -> 写入一个可 tail 的文件  : 引擎挂了的话，我们 PRINT 出原因，而不是盲目等到超时
import subprocess, time, urllib.request

LOG = "/content/vllm.log"
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
     "--max-model-len", "2048", "--port", "8000",
     "--enforce-eager", "--gpu-memory-utilization", "0.85"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(90):                       # 最多约 7.5 分钟（模型下载 + 加载）
    if server.poll() is not None:         # 进程已死 —— 打印真正的原因
        print(f"server exited (code {server.returncode}). Root cause in the log tail:\n")
        print("".join(open(LOG).readlines()[-25:]))
        break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("vLLM server is up on :8000"); up = True; break
    except Exception:
        time.sleep(5)
if not up and server.poll() is None:
    print("still starting (downloading / loading). Log tail:\n")
    print("".join(open(LOG).readlines()[-15:]))

# 常见故障："Free memory ... less than desired GPU memory utilization" 意味着有一个残留的 vLLM
# 仍然占着显卡。运行  !nvidia-smi  看它，然后  Runtime -> Restart session  再重跑。

In [ ]:
# 附录第 4 格 —— 点睛之笔：与第 01 节完全相同的客户端代码，指向你自己的服务器。
#
# 预期会发生什么：第 3 格只要 HTTP 层能应答 /v1/models 就会打印 "up"，但第一个
# 真正的请求仍可能撞上引擎最后的预热 —— 你可能会看到几秒钟的连接错误。
# 所以我们用退避重试，而不是第一次失败就放弃。如果服务器进程
# 真的死了（比如启动后引擎崩了），我们会检测到并打印日志尾部。
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")  # 本地 vLLM，不需要 key

resp = None
for attempt in range(1, 13):                      # 约 1 分钟的重试
    if 'server' in globals() and server.poll() is not None:   # 进程没了 -> 不是竞争，是崩溃
        print(f"server process exited (code {server.returncode}). Log tail:\n")
        print("".join(open("/content/vllm.log").readlines()[-25:]))
        break
    try:
        resp = client.chat.completions.create(
            model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
            messages=[{"role": "user", "content": "In one sentence, what does a serving framework do?"}],
            max_tokens=64)
        break                                     # 成功
    except Exception as e:
        print(f"attempt {attempt}: not ready yet ({type(e).__name__}) — retrying in 5s")
        time.sleep(5)

if resp:
    print("\n" + resp.choices[0].message.content)
    print("\nusage:", resp.usage)                 # 与第 00 节以来你一直在解析的响应结构相同
    # （TinyLlama 是 1.1B —— 别指望答案多好。重点在 SEAM，不在模型质量。）
else:
    print("\nno response — rerun cell 3 (it cleans up and relaunches), then rerun this cell.")

In [ ]:
# 附录第 5 格 —— 用完一定要释放 GPU。同一时刻只能有一个 vLLM 拥有这张卡；残留的
# 进程正是下次运行时报 "free memory" 错的根源。
server.terminate()
import time; time.sleep(3)
!pkill -9 -f vllm.entrypoints 2>/dev/null || true   # 双保险：杀掉任何游离的 worker
!nvidia-smi --query-gpu=memory.used --format=csv,noheader   # 应该回落到接近 ~0 MiB

# 注意 —— 释放显存不等于交出 GPU。这一格释放了内存，好让你
# 在这里重新启动 vLLM，但只要 runtime 还连着，你依然占着那块 T4，还在
# 消耗你的 Colab GPU 配额（免费档每天/每周都有限）。真正收工时：
#   Runtime -> Disconnect and delete runtime      （交还 GPU；释放配额）
# 之后再连上来会是一个全新的 runtime，所以上面的安装 + 启动格要重跑。

## 练习

1. **画出你自己的技术栈。** 拿出你在第 12 节计划中的 capstone 形态。写下来：用托管端点还是自托管？如果自托管，选四个框架里的哪一个、*为什么*，并点名决定它的那个指标。两句话；这正是面试官想听的答案。
2. **代码里的替换。** 什么都不用跑，写出把第 05 节的 agent 循环从 Groq 迁到自托管 vLLM 服务器的那两行 diff。确认循环里其他任何地方都没变，并用一句话说明为什么。
3. **是层次，不是排名。** 用你自己的话解释：为什么「Triton vs TensorRT-LLM」是类别错误，而「vLLM vs TGI」是公平比较。（提示：哪个是*服务器*，哪个是*引擎*？）
4. **（如果你跑了附录。）** 把附录里的模型换成第二个小模型，重启服务器，用同一个客户端打它。记下你的代码里*没变*的是什么：这种不变性正是 OpenAI 兼容这层缝隙的全部意义。